# Week 21 workbook --- reward hacking and scalable oversight

Eight graded exercises. Each scaffold cell is followed by a hidden
test cell that delegates to `project.solutions.exercise_N`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)

import numpy as np
import matplotlib.pyplot as plt

from project import solutions as S
from project.cai import critique, revise, compliance_score, dpo_loss
from project.hack_lab import proxy_reward, hack_train

## Exercise 1 ($\bigstar$). Classify a Goodhart scenario.

Given one of the canonical scenario tags (`noise_overfitting`,
`out_of_distribution`, `spurious_correlation`, `active_gaming`),
return the corresponding Manheim-Garrabrant variant
(`regressional`, `extremal`, `causal`, `adversarial`).

> **Hint**: noise is regressional; OOD is extremal; spurious is causal; gaming is adversarial.

In [ ]:
def exercise_1(scenario):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
for tag, expected in [('noise_overfitting', 'regressional'),
                       ('active_gaming', 'adversarial')]:
    assert S.exercise_1(tag) == expected
print('exercise 1 ok')

## Exercise 2 ($\bigstar\bigstar$). Proxy vs true quality disagreement.

Implement `proxy_reward` and `true_quality` (see `project/hack_lab.py`)
and compute their rank correlation on a curated text set. Find
a text where they actively disagree --- proxy high, truth low.

In [ ]:
def exercise_2(texts):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
texts = ['the cat sat on the mat',
         'good good good good good',
         'qweqweqwerlkjlkj',
         'the dog ran across the field']
px, tx, corr = S.exercise_2(texts)
print('proxy:', px)
print('truth:', tx)
print('rank corr:', corr)
assert px[1] > px[0]  # gameable text scores higher on proxy
assert tx[0] > tx[1]  # but worse on truth
print('exercise 2 ok')

## Exercise 3 ($\bigstar\bigstar$). KL coefficient schedule.

Linear decay, clamped at zero.

In [ ]:
def exercise_3(step, beta0=0.04, decay=0.001):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
assert S.exercise_3(0) == 0.04
assert S.exercise_3(1000) == 0.0
print('exercise 3 ok')

## Exercise 4 ($\bigstar\bigstar$). Length penalty.

In [ ]:
def exercise_4(reward, length, lam=0.01):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
assert abs(S.exercise_4(1.0, 100, lam=0.02) - (-1.0)) < 1e-9
print('exercise 4 ok')

## Exercise 5 ($\bigstar\bigstar$). Constitutional critique + revise.

Given a text, return (a) the per-principle pass/fail map, (b) the
revised text, (c) the compliance score before, (d) the compliance
score after.

In [ ]:
def exercise_5(text):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
crit, rev, c_pre, c_post = S.exercise_5('HELLO Xworld')
assert c_post >= c_pre
print(f'rev: {rev!r}  pre={c_pre} post={c_post}')
print('exercise 5 ok')

## Exercise 6 ($\bigstar\bigstar$). DPO loss in NumPy.

Implement the DPO loss without PyTorch and verify it matches the
reference torch version.

In [ ]:
def exercise_6(pol_c, pol_r, ref_c, ref_r, beta=0.1):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
import torch
rng = np.random.default_rng(0)
a, b, c, d = (rng.normal(size=8) for _ in range(4))
L_np = S.exercise_6(a, b, c, d, 0.1)
L_t = float(dpo_loss(*[torch.tensor(x, dtype=torch.float32) for x in (a, b, c, d)], beta=0.1))
assert abs(L_np - L_t) < 1e-4
print(f'exercise 6 ok  L_np={L_np:.6f}  L_t={L_t:.6f}')

## Exercise 7 ($\bigstar\bigstar$). Toy debate equilibrium.

Probability the truthful side wins with a $p$-accurate judge over $n$ rounds.

In [ ]:
def exercise_7(p, n):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
assert S.exercise_7(1.0, 1) == 1.0
assert S.exercise_7(0.6, 1) < S.exercise_7(0.6, 5)
print('exercise 7 ok')

## Exercise 8 ($\bigstar\bigstar\bigstar$, open-ended). Divergence index.

Run the hack-lab with and without each regulariser and compute the
divergence index from the resulting histories. Write 2-3 paragraphs
of analysis in the markdown cell below.

In [ ]:
def exercise_8(history, window=20):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
hist = {'proxy': [0.0]*20 + [1.0]*20, 'true_quality': [-2.0]*20 + [-3.0]*20}
assert abs(S.exercise_8(hist, window=20) - 2.0) < 1e-6
print('exercise 8 ok')

## Visualisation

Run the hack-lab once and plot the proxy / truth divergence.

In [ ]:
res = hack_train(seed=0, num_steps=40, sft_warmup_steps=80, batch=8, max_new_tokens=20)
h = res['history']
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax1.plot(h['step'], h['proxy'], color=RL_PALETTE['orange'], label='proxy')
ax2.plot(h['step'], h['true_quality'], color=RL_PALETTE['blue'], label='truth')
ax1.set_xlabel('step'); ax1.set_ylabel('proxy', color=RL_PALETTE['orange'])
ax2.set_ylabel('truth', color=RL_PALETTE['blue']); ax2.grid(False)
ax1.set_title('Proxy vs true quality (in vivo Goodhart)')
fig.tight_layout(); plt.show()

## Discussion (write your own)

1. Which of the three mitigations (KL anchor, length penalty, entropy bonus) most reliably preserved true quality? Why?
2. The Anthropic 2025 result shows reward-hacking *generalises* to unrelated misalignments. Sketch how that would manifest in a candidate-ranking system: if your ranker learns to game `interview_2_reached`, what other proxies would you expect to see degrade?
3. The CAI loop here uses a *deterministic* constitution. In the real Bai et al. recipe, an LLM does the critique and revision. What goes wrong if the critic LLM is biased in the same direction as the original SFT model?